### Loading the dataset

The data set was generated using the `simulated.py` script

In [1]:
import pandas as pd
import numpy as np

# Loading dataset into dataframe
df = pd.read_csv('../data/simulated_server_metrics.csv', parse_dates=['timestamp'])

df.head()

,timestamp,server_id,server_type,cpu_percent,memory_percent,disk_io,is_anomaly,anomaly_type
0,2025-01-01 00:00:00,web_1,web,24.585207,33.194309,51.865857,0,normal
1,2025-01-01 00:05:00,web_1,web,26.946724,37.129782,48.890155,0,normal
2,2025-01-01 00:10:00,web_1,web,29.583721,35.496424,45.627445,0,normal
3,2025-01-01 00:15:00,web_1,web,24.330454,30.116072,50.508945,0,normal
4,2025-01-01 00:20:00,web_1,web,24.356086,32.283149,47.617581,0,normal


### Computing the rolling averages
For each row the average of the last N readings are computed for each server. This smooths out noise and shows the recent trends. 

Rolling windows of 5, 10 and 30 are created to capture different times. For example 5 readings = 25 min "very recent", 30 readings = 2.5hrs "much longer".
`min_periods=1` is used for the first 4 rows which would have been `NaN`, this is a tiny fraction of the whole dataset is affected so the issue of less meaningful rows is negliable. 

`.shift` - ensures that the current observation isn't used to calculate the rolling value. For example we have a `cpu_percent` spike **95**, if this was the current observation and included in the rolling value, it could have made the anomaly signal weak and blend into the normal data. This needs to be done for each server type, and ensure that the values used to calculate the rolling mean are not from other server types

In [2]:
# Creating rolling windows for the mean and standard deviation
def add_rolling_features(df,column, windows):
    for window in windows:
        mean_col_name = f'{column}_roll_mean_{window}'
        shifted_values = df.groupby('server_id')[column].shift(1)
        df[mean_col_name] = shifted_values.groupby(df['server_id']).rolling(window=window, min_periods=1).mean().reset_index(level=0, drop=True)
        std_col_name = f'{column}_roll_standard_deviation_{window}'
        df[std_col_name] = shifted_values.groupby(df['server_id']).rolling(window=window, min_periods=1).std().reset_index(level=0, drop=True)
    return df

df = add_rolling_features(df, 'cpu_percent',[5,10,30])
df = add_rolling_features(df, 'memory_percent',[5,10,30])
df = add_rolling_features(df, 'disk_io',[5,10,30])
df.head(30)

,timestamp,server_id,server_type,cpu_percent,memory_percent,disk_io,is_anomaly,anomaly_type,cpu_percent_roll_mean_5,cpu_percent_roll_standard_deviation_5,...,memory_percent_roll_mean_10,memory_percent_roll_standard_deviation_10,memory_percent_roll_mean_30,memory_percent_roll_standard_deviation_30,disk_io_roll_mean_5,disk_io_roll_standard_deviation_5,disk_io_roll_mean_10,disk_io_roll_standard_deviation_10,disk_io_roll_mean_30,disk_io_roll_standard_deviation_30
0,2025-01-01 00:00:00,web_1,web,24.585207,33.194309,51.865857,0,normal,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-01 00:05:00,web_1,web,26.946724,37.129782,48.890155,0,normal,24.585207,NaN,...,33.194309,NaN,33.194309,NaN,51.865857,NaN,51.865857,NaN,51.865857,NaN
2,2025-01-01 00:10:00,web_1,web,29.583721,35.496424,45.627445,0,normal,25.765965,1.669845,...,35.162045,2.782800,35.162045,2.782800,50.378006,2.104139,50.378006,2.104139,50.378006,2.104139
3,2025-01-01 00:15:00,web_1,web,24.330454,30.116072,50.508945,0,normal,27.038551,2.500522,...,35.273505,1.977184,35.273505,1.977184,48.794486,3.120306,48.794486,3.120306,48.794486,3.120306
4,2025-01-01 00:20:00,web_1,web,24.356086,32.283149,47.617581,0,normal,26.361526,2.449868,...,33.984147,3.042359,33.984147,3.042359,49.223100,2.688069,49.223100,2.688069,49.223100,2.688069
5,2025-01-01 00:25:00,web_1,web,29.829008,34.611279,45.449479,0,normal,25.960438,2.303421,...,33.643947,2.742379,33.643947,2.742379,48.901997,2.436150,48.901997,2.436150,48.901997,2.436150
6,2025-01-01 00:30:00,web_1,web,27.433818,38.424553,47.917807,0,normal,27.009198,2.683192,...,33.805169,2.484445,33.805169,2.484445,47.618721,2.158746,48.326577,2.595093,48.326577,2.595093
7,2025-01-01 00:35:00,web_1,web,23.770490,36.510445,47.274504,0,normal,27.106617,2.689192,...,34.465081,2.862186,34.465081,2.862186,47.424251,2.056973,48.268181,2.374018,48.268181,2.374018
8,2025-01-01 00:40:00,web_1,web,26.861223,31.653193,48.527026,0,normal,25.943971,2.605858,...,34.720751,2.746771,34.720751,2.746771,47.753663,1.814931,48.143972,2.225814,48.143972,2.225814
9,2025-01-01 00:45:00,web_1,web,23.905126,34.775971,46.613106,0,normal,26.450125,2.455552,...,34.379912,2.765357,34.379912,2.765357,47.357279,1.161552,48.186533,2.085970,48.186533,2.085970


In [3]:
# Check the anomaly cpu_percent rolling mean and std
print(
    df[(df['server_id'] == 'web_1') & (df['is_anomaly'] == 1)]
    [['timestamp', 'server_id','cpu_percent', 'is_anomaly', 'cpu_percent_roll_mean_5','cpu_percent_roll_standard_deviation_5']]
    .tail()
)
# Test whether the calculated rolling means and std are within the server types, don't overlap with eachother 
print(
    df[df['server_id'] == 'web_2']
    [['timestamp', 'server_id', 'cpu_percent', 'is_anomaly', 'cpu_percent_roll_mean_5','cpu_percent_roll_standard_deviation_5']]
    .head()
)


               timestamp server_id  cpu_percent  is_anomaly  \
2732 2025-01-10 11:40:00     web_1    89.638834           1   
2733 2025-01-10 11:45:00     web_1    92.598409           1   
2734 2025-01-10 11:50:00     web_1    91.900169           1   
2735 2025-01-10 11:55:00     web_1    96.153894           1   
2736 2025-01-10 12:00:00     web_1    91.608226           1   

      cpu_percent_roll_mean_5  cpu_percent_roll_standard_deviation_5  
2732                92.854232                               2.590263  
2733                92.000210                               2.846816  
2734                92.695570                               2.349006  
2735                91.947582                               1.676026  
2736                92.334549                               2.399010  
                timestamp server_id  cpu_percent  is_anomaly  \
12096 2025-01-01 00:00:00     web_2    25.568484           0   
12097 2025-01-01 00:05:00     web_2    26.855105           0   
120

### Computing Z-score

Z-score is the number of standard deviations a data point is from the mean. 

Computed by the following formula :
$$
z = \frac{x - \mu}{\sigma}
$$

`x` = The raw data point,
$\mu$ = The mean ( average )of the dataset,
$\sigma$ = The standard deviation of the set


In [4]:
def compute_z_score(df, column, windows):
    for window in windows:
        column_name = f'{column}_zscore_{window}'
        roll_std_column_name = f'{column}_roll_standard_deviation_{window}'
        roll_mean_column_name = f'{column}_roll_mean_{window}'
        df[column_name] = np.where(df[roll_std_column_name] == 0, 0, (df[column] - df[roll_mean_column_name])/df[roll_std_column_name])
    return df
    
    

df = compute_z_score(df, 'cpu_percent',[5,10,30])
df = compute_z_score(df, 'memory_percent',[5,10,30])
df = compute_z_score(df, 'disk_io',[5,10,30])

df.head(10)

,timestamp,server_id,server_type,cpu_percent,memory_percent,disk_io,is_anomaly,anomaly_type,cpu_percent_roll_mean_5,cpu_percent_roll_standard_deviation_5,...,disk_io_roll_standard_deviation_30,cpu_percent_zscore_5,cpu_percent_zscore_10,cpu_percent_zscore_30,memory_percent_zscore_5,memory_percent_zscore_10,memory_percent_zscore_30,disk_io_zscore_5,disk_io_zscore_10,disk_io_zscore_30
0,2025-01-01 00:00:00,web_1,web,24.585207,33.194309,51.865857,0,normal,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-01 00:05:00,web_1,web,26.946724,37.129782,48.890155,0,normal,24.585207,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-01-01 00:10:00,web_1,web,29.583721,35.496424,45.627445,0,normal,25.765965,1.669845,...,2.104139,2.286294,2.286294,2.286294,0.120159,0.120159,0.120159,-2.257722,-2.257722,-2.257722
3,2025-01-01 00:15:00,web_1,web,24.330454,30.116072,50.508945,0,normal,27.038551,2.500522,...,3.120306,-1.083013,-1.083013,-1.083013,-2.608474,-2.608474,-2.608474,0.549452,0.549452,0.549452
4,2025-01-01 00:20:00,web_1,web,24.356086,32.283149,47.617581,0,normal,26.361526,2.449868,...,2.688069,-0.818591,-0.818591,-0.818591,-0.559105,-0.559105,-0.559105,-0.597276,-0.597276,-0.597276
5,2025-01-01 00:25:00,web_1,web,29.829008,34.611279,45.449479,0,normal,25.960438,2.303421,...,2.436150,1.679489,1.679489,1.679489,0.352735,0.352735,0.352735,-1.417203,-1.417203,-1.417203
6,2025-01-01 00:30:00,web_1,web,27.433818,38.424553,47.917807,0,normal,27.009198,2.683192,...,2.595093,0.158252,0.319198,0.319198,1.630920,1.859322,1.859322,0.138546,-0.157517,-0.157517
7,2025-01-01 00:35:00,web_1,web,23.770490,36.510445,47.274504,0,normal,27.106617,2.689192,...,2.374018,-1.240569,-1.235412,-1.235412,0.734515,0.714616,0.714616,-0.072800,-0.418563,-0.418563
8,2025-01-01 00:40:00,web_1,web,26.861223,31.653193,48.527026,0,normal,25.943971,2.605858,...,2.225814,0.351996,0.207107,0.207107,-0.829366,-1.116787,-1.116787,0.426111,0.172096,0.172096
9,2025-01-01 00:45:00,web_1,web,23.905126,34.775971,46.613106,0,normal,26.450125,2.455552,...,2.085970,-1.036426,-1.091700,-1.091700,0.027966,0.143222,0.143222,-0.640672,-0.754291,-0.754291


In [5]:
print(
    df[(df['server_id'] == 'web_1') & (df['is_anomaly'] == 1)]
    [['timestamp', 'server_id','cpu_percent', 'is_anomaly', 'cpu_percent_roll_mean_5','cpu_percent_roll_standard_deviation_5','cpu_percent_zscore_5']]
    .head()
)

print(
    df[df['server_id'] == 'web_2']
    [['timestamp', 'server_id', 'cpu_percent', 'is_anomaly', 'cpu_percent_roll_mean_5','cpu_percent_roll_standard_deviation_5','cpu_percent_zscore_5']]
    .head()
)

               timestamp server_id  cpu_percent  is_anomaly  \
2712 2025-01-10 10:00:00     web_1    89.280091           1   
2713 2025-01-10 10:05:00     web_1    90.413260           1   
2714 2025-01-10 10:10:00     web_1    91.156097           1   
2715 2025-01-10 10:15:00     web_1    95.267062           1   
2716 2025-01-10 10:20:00     web_1    91.295033           1   

      cpu_percent_roll_mean_5  cpu_percent_roll_standard_deviation_5  \
2712                54.603730                               3.322822   
2713                62.116670                              15.460035   
2714                68.668278                              19.507853   
2715                76.759689                              18.558552   
2716                84.226160                              16.485458   

      cpu_percent_zscore_5  
2712             10.435815  
2713              1.830306  
2714              1.152757  
2715              0.997242  
2716              0.428794  
              

In [6]:
df[
    (df['server_id'] == 'web_1') &
    (df['timestamp'] >= '2025-01-10 09:50:00') &
    (df['timestamp'] <= '2025-01-10 11:50:00')
][[
    'timestamp',
    'cpu_percent',
    'is_anomaly',
    'cpu_percent_roll_mean_5',
    'cpu_percent_roll_standard_deviation_5',
    'cpu_percent_zscore_5'
]]

,timestamp,cpu_percent,is_anomaly,cpu_percent_roll_mean_5,cpu_percent_roll_standard_deviation_5,cpu_percent_zscore_5
2710,2025-01-10 09:50:00,57.934704,0,53.101151,2.680283,1.803374
2711,2025-01-10 09:55:00,55.014291,0,54.128892,3.417767,0.259058
2712,2025-01-10 10:00:00,89.280091,1,54.603730,3.322822,10.435815
2713,2025-01-10 10:05:00,90.413260,1,62.116670,15.460035,1.830306
2714,2025-01-10 10:10:00,91.156097,1,68.668278,19.507853,1.152757
2715,2025-01-10 10:15:00,95.267062,1,76.759689,18.558552,0.997242
2716,2025-01-10 10:20:00,91.295033,1,84.226160,16.485458,0.428794
2717,2025-01-10 10:25:00,93.027524,1,91.482309,2.261552,0.683254
2718,2025-01-10 10:30:00,87.800975,1,92.231795,1.948573,-2.273879
2719,2025-01-10 10:35:00,93.547604,1,91.709338,2.745362,0.669590


### Computing time-based features

In [7]:
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek

df.head(10)

,timestamp,server_id,server_type,cpu_percent,memory_percent,disk_io,is_anomaly,anomaly_type,cpu_percent_roll_mean_5,cpu_percent_roll_standard_deviation_5,...,cpu_percent_zscore_10,cpu_percent_zscore_30,memory_percent_zscore_5,memory_percent_zscore_10,memory_percent_zscore_30,disk_io_zscore_5,disk_io_zscore_10,disk_io_zscore_30,hour,day_of_week
0,2025-01-01 00:00:00,web_1,web,24.585207,33.194309,51.865857,0,normal,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,2
1,2025-01-01 00:05:00,web_1,web,26.946724,37.129782,48.890155,0,normal,24.585207,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,2
2,2025-01-01 00:10:00,web_1,web,29.583721,35.496424,45.627445,0,normal,25.765965,1.669845,...,2.286294,2.286294,0.120159,0.120159,0.120159,-2.257722,-2.257722,-2.257722,0,2
3,2025-01-01 00:15:00,web_1,web,24.330454,30.116072,50.508945,0,normal,27.038551,2.500522,...,-1.083013,-1.083013,-2.608474,-2.608474,-2.608474,0.549452,0.549452,0.549452,0,2
4,2025-01-01 00:20:00,web_1,web,24.356086,32.283149,47.617581,0,normal,26.361526,2.449868,...,-0.818591,-0.818591,-0.559105,-0.559105,-0.559105,-0.597276,-0.597276,-0.597276,0,2
5,2025-01-01 00:25:00,web_1,web,29.829008,34.611279,45.449479,0,normal,25.960438,2.303421,...,1.679489,1.679489,0.352735,0.352735,0.352735,-1.417203,-1.417203,-1.417203,0,2
6,2025-01-01 00:30:00,web_1,web,27.433818,38.424553,47.917807,0,normal,27.009198,2.683192,...,0.319198,0.319198,1.630920,1.859322,1.859322,0.138546,-0.157517,-0.157517,0,2
7,2025-01-01 00:35:00,web_1,web,23.770490,36.510445,47.274504,0,normal,27.106617,2.689192,...,-1.235412,-1.235412,0.734515,0.714616,0.714616,-0.072800,-0.418563,-0.418563,0,2
8,2025-01-01 00:40:00,web_1,web,26.861223,31.653193,48.527026,0,normal,25.943971,2.605858,...,0.207107,0.207107,-0.829366,-1.116787,-1.116787,0.426111,0.172096,0.172096,0,2
9,2025-01-01 00:45:00,web_1,web,23.905126,34.775971,46.613106,0,normal,26.450125,2.455552,...,-1.091700,-1.091700,0.027966,0.143222,0.143222,-0.640672,-0.754291,-0.754291,0,2


### Rate of change 

Essentially calculating how much did this value change from the previous reading for each `server_id`.

In [8]:
def calculate_rate_of_change(df,column):
    column_name = f'{column}_rate_of_change'
    df[column_name] = df.groupby('server_id')[column].diff()
    return df

df = calculate_rate_of_change(df,'cpu_percent')
df = calculate_rate_of_change(df,'memory_percent')
df = calculate_rate_of_change(df,'disk_io')

print(
    df[(df['server_id'] == 'web_1') & (df['is_anomaly'] == 1)]
    [['timestamp', 'server_id','cpu_percent', 'is_anomaly','cpu_percent_rate_of_change']]
    .head()
)

               timestamp server_id  cpu_percent  is_anomaly  \
2712 2025-01-10 10:00:00     web_1    89.280091           1   
2713 2025-01-10 10:05:00     web_1    90.413260           1   
2714 2025-01-10 10:10:00     web_1    91.156097           1   
2715 2025-01-10 10:15:00     web_1    95.267062           1   
2716 2025-01-10 10:20:00     web_1    91.295033           1   

      cpu_percent_rate_of_change  
2712                   34.265799  
2713                    1.133169  
2714                    0.742838  
2715                    4.110965  
2716                   -3.972029  


### Remove NaN values
Standard deviation is calculated by 

$$s = \sqrt{\frac{\sum (x_i - \bar{x})^2}{n - 1}}$$

and requires atleast 2 values to be computed, we have 15 servers, exactly 15 first rows and 15 NaN's consistently across the `std` and `Z-score`. Below we drop these rows.

We also have NaN values the `cpu_percent_rate_of_change`, `memory_percent_rate_of_change` and `disk_io_rate_of_change`, on the first row for every `server_id`

In [9]:
df.isna().sum()

timestamp                                     0
server_id                                     0
server_type                                   0
cpu_percent                                   0
memory_percent                                0
disk_io                                       0
is_anomaly                                    0
anomaly_type                                  0
cpu_percent_roll_mean_5                      15
cpu_percent_roll_standard_deviation_5        30
cpu_percent_roll_mean_10                     15
cpu_percent_roll_standard_deviation_10       30
cpu_percent_roll_mean_30                     15
cpu_percent_roll_standard_deviation_30       30
memory_percent_roll_mean_5                   15
memory_percent_roll_standard_deviation_5     30
memory_percent_roll_mean_10                  15
memory_percent_roll_standard_deviation_10    30
memory_percent_roll_mean_30                  15
memory_percent_roll_standard_deviation_30    30
disk_io_roll_mean_5                     

In [10]:
print(len(df))
df = df.dropna()
print(len(df))
df.isna().sum()

181440
181410


timestamp                                    0
server_id                                    0
server_type                                  0
cpu_percent                                  0
memory_percent                               0
disk_io                                      0
is_anomaly                                   0
anomaly_type                                 0
cpu_percent_roll_mean_5                      0
cpu_percent_roll_standard_deviation_5        0
cpu_percent_roll_mean_10                     0
cpu_percent_roll_standard_deviation_10       0
cpu_percent_roll_mean_30                     0
cpu_percent_roll_standard_deviation_30       0
memory_percent_roll_mean_5                   0
memory_percent_roll_standard_deviation_5     0
memory_percent_roll_mean_10                  0
memory_percent_roll_standard_deviation_10    0
memory_percent_roll_mean_30                  0
memory_percent_roll_standard_deviation_30    0
disk_io_roll_mean_5                          0
disk_io_roll_

### Adding server type
This would be an encoded categorical feature, this will allow the model to account for type differences.

In [11]:
df = pd.get_dummies(df,columns=['server_type'],dtype=int)
df.head()

,timestamp,server_id,cpu_percent,memory_percent,disk_io,is_anomaly,anomaly_type,cpu_percent_roll_mean_5,cpu_percent_roll_standard_deviation_5,cpu_percent_roll_mean_10,...,hour,day_of_week,cpu_percent_rate_of_change,memory_percent_rate_of_change,disk_io_rate_of_change,server_type_batch_worker,server_type_cache,server_type_database,server_type_load_balancer,server_type_web
2,2025-01-01 00:10:00,web_1,29.583721,35.496424,45.627445,0,normal,25.765965,1.669845,25.765965,...,0,2,2.636997,-1.633358,-3.262710,0,0,0,0,1
3,2025-01-01 00:15:00,web_1,24.330454,30.116072,50.508945,0,normal,27.038551,2.500522,27.038551,...,0,2,-5.253267,-5.380352,4.881500,0,0,0,0,1
4,2025-01-01 00:20:00,web_1,24.356086,32.283149,47.617581,0,normal,26.361526,2.449868,26.361526,...,0,2,0.025633,2.167077,-2.891364,0,0,0,0,1
5,2025-01-01 00:25:00,web_1,29.829008,34.611279,45.449479,0,normal,25.960438,2.303421,25.960438,...,0,2,5.472921,2.328130,-2.168102,0,0,0,0,1
6,2025-01-01 00:30:00,web_1,27.433818,38.424553,47.917807,0,normal,27.009198,2.683192,26.605200,...,0,2,-2.395189,3.813275,2.468328,0,0,0,0,1


### Save the new dataset

In [12]:
df.to_csv('../data/features_rolling.csv', index=False)

## Rolling window limitation: sustained anomalies get absorbed into their own baseline

After training an isolation forest model using the rolling window features, I had found that these features don't strongly represent anomalies.

Example: 
When comparing `cpu_zscore_5` against `cpu_zscore_30` for `web_1`'s injected 2-hour CPU spike
(2025-01-10, 10:00–12:00), a clear limitation of rolling-window-based Z-scores becomes visible.

With a short window (5), the Z-score is highest at the very start of the anomaly (~1.5), but
drops toward zero — and even goes negative — within about 25 minutes. This happens because the
rolling mean/std are recalculated from the last 5 readings, which very quickly become entirely
made up of anomalous values. The window "absorbs" the anomaly and starts treating it as the new
normal, even though the underlying issue is still ongoing.

A longer window (30) is more resistant to this — the Z-score stays positive and clearly elevated
(roughly 0.3–2.6) for the full 2-hour window, since the anomaly can't fully saturate 30 readings
within that time. However, even this longer window shows a gradual decline in Z-score over the
duration of the anomaly, as more anomalous readings enter the window.

**Takeaway:** short rolling windows are effective at flagging the *onset* of a sustained anomaly,
but rapidly lose sensitivity the longer that anomaly continues, because the baseline they compare
against is itself built from recent — and increasingly contaminated — data. Longer windows delay
this effect but don't eliminate it.

**Next experiment**: generate baselines and residuals for `cpu_percent`, `memory_percent` and `disk_io`.

In [13]:
df[(df['server_id'] == 'web_1') & (df['is_anomaly'] == 1)][['timestamp', 'cpu_percent', 'cpu_percent_roll_mean_30', 'cpu_percent_roll_standard_deviation_30', 'cpu_percent_zscore_30']]

,timestamp,cpu_percent,cpu_percent_roll_mean_30,cpu_percent_roll_standard_deviation_30,cpu_percent_zscore_30
2712,2025-01-10 10:00:00,89.280091,51.418177,3.751468,10.092559
2713,2025-01-10 10:05:00,90.413260,52.779148,7.828517,4.807310
2714,2025-01-10 10:10:00,91.156097,54.207292,10.347875,3.570666
2715,2025-01-10 10:15:00,95.267062,55.651905,12.271438,3.228241
2716,2025-01-10 10:20:00,91.295033,57.208159,14.158878,2.407456
2717,2025-01-10 10:25:00,93.027524,58.822524,15.190420,2.251748
2718,2025-01-10 10:30:00,87.800975,60.255544,16.318919,1.687945
2719,2025-01-10 10:35:00,93.547604,61.519037,16.944584,1.890195
2720,2025-01-10 10:40:00,86.672905,63.127379,17.627889,1.335697
2721,2025-01-10 10:45:00,91.779746,64.419810,17.892338,1.529143


In [14]:
df[(df['server_id'] == 'web_1') & (df['is_anomaly'] == 1)][['timestamp', 'cpu_percent', 'cpu_percent_roll_mean_5', 'cpu_percent_roll_standard_deviation_5', 'cpu_percent_zscore_5']]

,timestamp,cpu_percent,cpu_percent_roll_mean_5,cpu_percent_roll_standard_deviation_5,cpu_percent_zscore_5
2712,2025-01-10 10:00:00,89.280091,54.603730,3.322822,10.435815
2713,2025-01-10 10:05:00,90.413260,62.116670,15.460035,1.830306
2714,2025-01-10 10:10:00,91.156097,68.668278,19.507853,1.152757
2715,2025-01-10 10:15:00,95.267062,76.759689,18.558552,0.997242
2716,2025-01-10 10:20:00,91.295033,84.226160,16.485458,0.428794
2717,2025-01-10 10:25:00,93.027524,91.482309,2.261552,0.683254
2718,2025-01-10 10:30:00,87.800975,92.231795,1.948573,-2.273879
2719,2025-01-10 10:35:00,93.547604,91.709338,2.745362,0.669590
2720,2025-01-10 10:40:00,86.672905,92.187639,2.831842,-1.947402
2721,2025-01-10 10:45:00,91.779746,90.468808,3.091718,0.424016


## Reload the simulated dataset

In [15]:
import pandas as pd
import numpy as np

# Loading dataset into dataframe
df = pd.read_csv('../data/simulated_server_metrics.csv', parse_dates=['timestamp'])

df.head()

,timestamp,server_id,server_type,cpu_percent,memory_percent,disk_io,is_anomaly,anomaly_type
0,2025-01-01 00:00:00,web_1,web,24.585207,33.194309,51.865857,0,normal
1,2025-01-01 00:05:00,web_1,web,26.946724,37.129782,48.890155,0,normal
2,2025-01-01 00:10:00,web_1,web,29.583721,35.496424,45.627445,0,normal
3,2025-01-01 00:15:00,web_1,web,24.330454,30.116072,50.508945,0,normal
4,2025-01-01 00:20:00,web_1,web,24.356086,32.283149,47.617581,0,normal


### Computing time-based features

In [16]:
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek

df.head(10)

,timestamp,server_id,server_type,cpu_percent,memory_percent,disk_io,is_anomaly,anomaly_type,hour,day_of_week
0,2025-01-01 00:00:00,web_1,web,24.585207,33.194309,51.865857,0,normal,0,2
1,2025-01-01 00:05:00,web_1,web,26.946724,37.129782,48.890155,0,normal,0,2
2,2025-01-01 00:10:00,web_1,web,29.583721,35.496424,45.627445,0,normal,0,2
3,2025-01-01 00:15:00,web_1,web,24.330454,30.116072,50.508945,0,normal,0,2
4,2025-01-01 00:20:00,web_1,web,24.356086,32.283149,47.617581,0,normal,0,2
5,2025-01-01 00:25:00,web_1,web,29.829008,34.611279,45.449479,0,normal,0,2
6,2025-01-01 00:30:00,web_1,web,27.433818,38.424553,47.917807,0,normal,0,2
7,2025-01-01 00:35:00,web_1,web,23.770490,36.510445,47.274504,0,normal,0,2
8,2025-01-01 00:40:00,web_1,web,26.861223,31.653193,48.527026,0,normal,0,2
9,2025-01-01 00:45:00,web_1,web,23.905126,34.775971,46.613106,0,normal,0,2


### Generating baselines & residual
- The **baseline** represents the typical behaviour of the specifc server at a specific hour
- The **residual** represents how far is the current resource usage from the expected behaviour

In [17]:
def generate_baselines_and_residuals (column):
    baseline_column_name = f'{column}_baseline'
    residual_column_name = f'{column}_residual'
    
    df[baseline_column_name] = (
        df.groupby(['server_id','hour'])[column]
            .transform('median')
    )
    
    df[residual_column_name] = (
        df[column] - df[baseline_column_name]
    )

    return df
                                 
df = generate_baselines_and_residuals("cpu_percent")
df = generate_baselines_and_residuals("memory_percent")
df = generate_baselines_and_residuals("disk_io")

In [18]:
df[
    (df['timestamp'] >= '2025-01-21 03:10:00') &
    (df['timestamp'] <= '2025-01-21 08:10:00') &
    (df['server_id'] == 'web_2')
][[
        'server_id',
        'is_anomaly', 
        'hour', 
        'cpu_percent', 
        'cpu_percent_baseline',
        'cpu_percent_residual'
]].head(20)

,server_id,is_anomaly,hour,cpu_percent,cpu_percent_baseline,cpu_percent_residual
17894,web_2,0,3,36.031807,30.219245,5.812562
17895,web_2,0,3,26.373275,30.219245,-3.845970
17896,web_2,0,3,31.739976,30.219245,1.520731
17897,web_2,0,3,30.752808,30.219245,0.533563
17898,web_2,0,3,30.457737,30.219245,0.238492
17899,web_2,0,3,28.626463,30.219245,-1.592782
17900,web_2,0,3,29.776246,30.219245,-0.442999
17901,web_2,0,3,30.829519,30.219245,0.610274
17902,web_2,0,3,33.830995,30.219245,3.611751
17903,web_2,0,3,35.195139,30.219245,4.975894


In [19]:
df[
    (df['timestamp'] >= '2025-01-15 13:00:00') &
    (df['timestamp'] <= '2025-01-15 16:00:00') &
    (df['server_id'] == 'database_1')
][[
        'server_id',
        'is_anomaly', 
        'hour', 
        'cpu_percent', 
        'cpu_percent_baseline',
        'cpu_percent_residual'
]].head(30)

,server_id,is_anomaly,hour,cpu_percent,cpu_percent_baseline,cpu_percent_residual
40476,database_1,0,13,73.979762,63.742903,10.236859
40477,database_1,0,13,63.224263,63.742903,-0.518639
40478,database_1,0,13,65.780770,63.742903,2.037867
40479,database_1,0,13,62.192557,63.742903,-1.550345
40480,database_1,0,13,59.124344,63.742903,-4.618558
40481,database_1,0,13,61.710581,63.742903,-2.032322
40482,database_1,0,13,67.231585,63.742903,3.488682
40483,database_1,0,13,68.439319,63.742903,4.696416
40484,database_1,0,13,61.576598,63.742903,-2.166305
40485,database_1,0,13,64.750297,63.742903,1.007394


### Adding server type

In [20]:
df = pd.get_dummies(df,columns=['server_type'],dtype=int)
df.head()

,timestamp,server_id,cpu_percent,memory_percent,disk_io,is_anomaly,anomaly_type,hour,day_of_week,cpu_percent_baseline,cpu_percent_residual,memory_percent_baseline,memory_percent_residual,disk_io_baseline,disk_io_residual,server_type_batch_worker,server_type_cache,server_type_database,server_type_load_balancer,server_type_web
0,2025-01-01 00:00:00,web_1,24.585207,33.194309,51.865857,0,normal,0,2,25.208847,-0.623640,34.838226,-1.643918,50.063459,1.802398,0,0,0,0,1
1,2025-01-01 00:05:00,web_1,26.946724,37.129782,48.890155,0,normal,0,2,25.208847,1.737876,34.838226,2.291556,50.063459,-1.173304,0,0,0,0,1
2,2025-01-01 00:10:00,web_1,29.583721,35.496424,45.627445,0,normal,0,2,25.208847,4.374873,34.838226,0.658198,50.063459,-4.436014,0,0,0,0,1
3,2025-01-01 00:15:00,web_1,24.330454,30.116072,50.508945,0,normal,0,2,25.208847,-0.878394,34.838226,-4.722154,50.063459,0.445486,0,0,0,0,1
4,2025-01-01 00:20:00,web_1,24.356086,32.283149,47.617581,0,normal,0,2,25.208847,-0.852761,34.838226,-2.555077,50.063459,-2.445878,0,0,0,0,1


In [21]:
# save the new dataset
df.to_csv('../data/features_residual.csv', index=False)